In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd 
import cartopy.crs as ccrs
import cmocean
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [2]:
from OceanDataStore import OceanDataCatalog

In [3]:
catalog = OceanDataCatalog(catalog_name="noc-model-stac")

In [4]:
catalog.available_collections

['noc-rapid-evolution', 'noc-npd-jra55', 'noc-npd-era5']

In [5]:
catalog.search(collection='noc-npd-era5', standard_name='sea_surface_temperature')


            * Item ID: noc-npd-era5/npd-eorca1-era5v1/gn/T1y
              Title: eORCA1 ERA5v1 NPD T1y Icechunk repository
              Description: Icechunk repository containing eORCA1 ERA5v1 NPD global ocean physics annual mean outputs defined at T-points.
              Platform: gn
              Start Date: 1976-01-01T00:00:00Z
              End Date: 2024-12-31T00:00:00Z
            

            * Item ID: noc-npd-era5/npd-eorca1-era5v1/gn/T1m
              Title: eORCA1 ERA5v1 NPD T1m Icechunk repository
              Description: Icechunk repository containing eORCA1 ERA5v1 NPD global ocean physics monthly mean outputs defined at T-points.
              Platform: gn
              Start Date: 1976-01-01T00:00:00Z
              End Date: 2024-12-31T00:00:00Z
            

            * Item ID: noc-npd-era5/npd-eorca025-era5v1/gn/T1y_3d
              Title: eORCA025 ERA5v1 NPD T1y_3d Icechunk repository
              Description: Icechunk repository containing eORCA025 ERA5v1 

In [6]:
catalog.available_items

['noc-npd-era5/npd-eorca1-era5v1/gn/T1y',
 'noc-npd-era5/npd-eorca1-era5v1/gn/T1m',
 'noc-npd-era5/npd-eorca025-era5v1/gn/T1y_3d',
 'noc-npd-era5/npd-eorca025-era5v1/gn/T1m_3d',
 'noc-npd-era5/npd-eorca025-era5v1/gn/T5d_3d',
 'noc-npd-era5/npd-eorca12-era5v1/gn/T1y_3d',
 'noc-npd-era5/npd-eorca12-era5v1/gn/T1m_3d']

In [7]:
catalog.Items[1]

<Item id=noc-npd-era5/npd-eorca12-era5v1/gn/T1m_3d>

In [10]:
ds1 = catalog.open_dataset(id=catalog.Items[1].id,
                          start_datetime='1976-01',
                          end_datetime='2024-12',                        
                          bbox = (-85.0, 0.0, 0.0, 80.0))

In [11]:
## Plotting Animation of September temperature anomaly

plt.close('all')
fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(10, 6))

septembers = []
for year in range (1976, 2025):
    septembers.append(f'{year}-09')

means = (ds1['tos_con'].sel(time_counter = septembers, method = 'nearest')).mean(dim = 'time_counter')
ds1['T_anomaly'] = ds1['tos_con'] - means

first_data = ds1['T_anomaly'].sel(time_counter=septembers[0], method='nearest') 
im = ax.pcolormesh(ds1['nav_lon'], ds1['nav_lat'], first_data, cmap='RdBu_r', transform=ccrs.PlateCarree(), vmin=-5, vmax=5)  
ax.set_extent([-85.0, 0.0, 0.0, 80.0], crs=ccrs.PlateCarree())
ax.coastlines()
ax.gridlines(draw_labels=True, linewidth=0.5, linestyle='--')
plt.colorbar(im, ax=ax, label='Temperature (°C)')
title = ax.set_title('Sea Surface Temperature Anomaly - September 1976')

def update(frame):
    ax.clear()
    data = ds1['T_anomaly'].sel(time_counter=septembers[frame], method='nearest')
    im = ax.pcolormesh(ds1['nav_lon'], ds1['nav_lat'], data, cmap='RdBu_r', transform=ccrs.PlateCarree(), vmin=-5, vmax=5)
    ax.set_extent([-85.0, 0.0, 0.0, 80.0], crs=ccrs.PlateCarree())
    ax.coastlines()
    ax.gridlines(draw_labels=True, linewidth=0.5, linestyle='--')
    ax.set_title(f'Sea Surface Temperature Anomaly - September {septembers[frame][:4]}')
    return [im]

plt.close()
anim = FuncAnimation(fig, update, frames=len(septembers), interval=1000, repeat=True)
HTML(anim.to_jshtml())


KeyboardInterrupt: 